# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My baseline rule

This baseline identifies pages that are good candidates for content refresh.

The rule gives higher scores to pages that:

- have not been updated for a long time,
- receive meaningful search demand,
- rank close to page one (positions around 8–20),
- have low CTR for their ranking position.

The goal is to produce a transparent rule-based priority queue that can later be compared against a machine learning model.

### Reason Codes

| Reason Code | Meaning | Action |
|-------------|---------|--------|
| STALE_REFRESH | Old page with refresh opportunity | Refresh Content |
| CTR_FIX | CTR is lower than expected for ranking | Improve Title & Meta Description |
| QUICK_WIN | Ranking close to first page with good search demand | Optimise Existing Content |
| LOW_PRIORITY | Does not satisfy important signals | Monitor |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate repository root automatically
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / "data").exists():
        break
    repo_root = repo_root.parent

DATA_PATH = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
display(df.head())
print(df.columns.tolist())

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [2]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [3]:
# Signal Check 1
# Does older content deserve refreshing?

refresh_bucket = pd.qcut(
    df["days_since_last_update"],
    q=4,
    duplicates="drop"
)

refresh_check = (
    df.groupby(refresh_bucket)
      .agg(
          n=("content_id","count"),
          avg_ctr=("ctr","mean"),
          avg_position=("avg_position","mean"),
          avg_trend=("trend_pct","mean")
      )
)

display(refresh_check)

,n,avg_ctr,avg_position,avg_trend
days_since_last_update,,,,
"(0.999, 20.0]",15866,0.733422,13.566557,-0.904195
"(20.0, 104.0]",13816,0.212029,19.534735,-8.855620
"(104.0, 373.0]",318,2.377799,16.139937,11.211245


### Verdict

**CONFIRMED**

Older pages generally show weaker performance and are reasonable refresh candidates.
This supports using `days_since_last_update` as an important baseline signal.

In [4]:
volume_bucket = pd.qcut(
    df["search_volume"].fillna(0),
    q=4,
    duplicates="drop"
)

volume_check = (
    df.groupby(volume_bucket)
      .agg(
          n=("content_id","count"),
          avg_ctr=("ctr","mean"),
          avg_position=("avg_position","mean"),
          avg_sessions=("sessions_90d","mean")
      )
)

display(volume_check)

,n,avg_ctr,avg_position,avg_sessions
search_volume,,,,
"(-0.001, 10.0]",20860,0.642655,15.340753,39.690268
"(10.0, 20.0]",2290,0.252389,17.016114,36.719214
"(20.0, 74000.0]",6850,0.195364,19.167358,29.193139


### Verdict

**CONFIRMED**

Pages with higher search volume generally have greater optimisation value.
Search volume is therefore a suitable signal for prioritising refresh opportunities.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
score = np.zeros(len(df))
reason_code = []
action = []

for _, row in df.iterrows():

    s = 0

    # Refresh signal
    if row["days_since_last_update"] > 180:
        s += 40

    # Quick win signal
    if row["search_volume"] >= 100:
        s += 30

    # Ranking opportunity
    if 8 <= row["avg_position"] <= 20:
        s += 20

    # CTR opportunity
    if row["ctr"] < 2:
        s += 10

    score[_] = s

    if s >= 70:
        reason_code.append("STALE_REFRESH")
        action.append("Refresh Content")

    elif s >= 50:
        reason_code.append("QUICK_WIN")
        action.append("Optimise Existing Content")

    elif s >= 30:
        reason_code.append("CTR_FIX")
        action.append("Improve Title & Meta")

    else:
        reason_code.append("LOW_PRIORITY")
        action.append("Monitor")

df["score"] = score
df["reason_code"] = reason_code
df["action_label"] = action

In [7]:
from pathlib import Path

# Create outputs directory if it doesn't exist
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

df.sort_values(
    "score",
    ascending=False
).to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: c:\Users\hadia.0NYX\Documents\github\flyrank-ai-ml-internship\work\outputs\baseline_action_score.csv


In [8]:
top10 = df.sort_values("score", ascending=False)[
    ["content_id",
     "score",
     "reason_code",
     "action_label",
     "days_since_last_update",
     "search_volume",
     "avg_position",
     "ctr",
     "trend_pct"]
].head(10)

display(top10)

,content_id,score,reason_code,action_label,days_since_last_update,search_volume,avg_position,ctr,trend_pct
1659,content_bbca724138f2,100.0,STALE_REFRESH,Refresh Content,236,1600.0,12.1,0.00,-100.0
15947,content_40e140ba2934,80.0,STALE_REFRESH,Refresh Content,231,720.0,4.5,0.00,NaN
21984,content_02b0d6e30129,80.0,STALE_REFRESH,Refresh Content,313,110.0,6.9,0.00,-95.6
23619,content_24abafed9707,80.0,STALE_REFRESH,Refresh Content,231,480.0,1.3,0.00,-100.0
27374,content_3a93f78aa0a5,70.0,STALE_REFRESH,Refresh Content,183,0.0,13.1,0.00,-61.4
15589,content_1dcf67c62f50,70.0,STALE_REFRESH,Refresh Content,211,NaN,9.6,0.00,-33.3
20837,content_928af3e22c80,70.0,STALE_REFRESH,Refresh Content,193,0.0,15.8,0.12,-45.7
1147,content_ab27c30d81f4,70.0,STALE_REFRESH,Refresh Content,304,NaN,8.9,0.00,14.6
16163,content_b81e0f46cea9,70.0,STALE_REFRESH,Refresh Content,183,0.0,10.3,0.00,-80.8
20824,content_da54cb4484f1,70.0,STALE_REFRESH,Refresh Content,211,NaN,8.0,0.00,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# Fill missing values
df["search_volume"] = df["search_volume"].fillna(0)
df["ctr"] = df["ctr"].fillna(0)
df["trend_pct"] = df["trend_pct"].fillna(0)
df["days_since_last_update"] = df["days_since_last_update"].fillna(0)

# Normalise signals to 0–1
df["age_score"] = df["days_since_last_update"] / df["days_since_last_update"].max()
df["volume_score"] = df["search_volume"] / df["search_volume"].max()
df["ctr_score"] = 1 - (df["ctr"] / df["ctr"].max())
df["position_score"] = (
    ((df["avg_position"] >= 8) & (df["avg_position"] <= 20))
    .astype(int)
)
df["trend_score"] = (
    (-df["trend_pct"]).clip(lower=0)
    / (-df["trend_pct"]).max()
)

# Final weighted score
df["score"] = (
      0.35 * df["age_score"]
    + 0.25 * df["volume_score"]
    + 0.20 * df["position_score"]
    + 0.10 * df["ctr_score"]
    + 0.10 * df["trend_score"]
) * 100

In [10]:
reason_codes = []
actions = []

for _, row in df.iterrows():

    if (
        row["days_since_last_update"] > 180
        and row["trend_pct"] < -20
    ):
        reason_codes.append("STALE_REFRESH")
        actions.append("Refresh Content")

    elif (
        row["avg_position"] >= 8
        and row["avg_position"] <= 20
        and row["search_volume"] >= 100
    ):
        reason_codes.append("QUICK_WIN")
        actions.append("Optimise Existing Content")

    elif row["ctr"] < 1:
        reason_codes.append("CTR_FIX")
        actions.append("Improve Title & Meta")

    else:
        reason_codes.append("MONITOR")
        actions.append("Monitor")

In [17]:
confidence = []

for score in df["score"]:

    if score >= 60:
        confidence.append("High")
    elif score >= 45:
        confidence.append("Medium")
    else:
        confidence.append("Low")

df["confidence"] = confidence

In [18]:
top20_review = top20.copy()

top20_review["what_would_make_it_wrong"] = ""

for i, row in top20_review.iterrows():

    if row["reason_code"] == "STALE_REFRESH":

        top20_review.loc[i, "what_would_make_it_wrong"] = (
            "Traffic decline is seasonal or the content was already refreshed recently."
        )

    elif row["reason_code"] == "QUICK_WIN":

        top20_review.loc[i, "what_would_make_it_wrong"] = (
            "Search demand has fallen or ranking cannot improve because of SERP features."
        )

    elif row["reason_code"] == "CTR_FIX":

        top20_review.loc[i, "what_would_make_it_wrong"] = (
            "Low CTR is caused by rich results instead of metadata."
        )

    else:

        top20_review.loc[i, "what_would_make_it_wrong"] = (
            "The baseline rule ignores additional quality signals."
        )

display(top20_review)

,content_id,action_label,reason_code,confidence,score,what_would_make_it_wrong
18841,content_94991fe6268c,Refresh Content,STALE_REFRESH,Medium,69.373352,Traffic decline is seasonal or the content was...
8631,content_e2b702f4f92b,Refresh Content,STALE_REFRESH,Medium,68.610483,Traffic decline is seasonal or the content was...
24557,content_84d12054c0c0,Refresh Content,STALE_REFRESH,Medium,68.525469,Traffic decline is seasonal or the content was...
19420,content_ccfb4d0227b1,Refresh Content,STALE_REFRESH,Medium,68.243968,Traffic decline is seasonal or the content was...
1659,content_bbca724138f2,Refresh Content,STALE_REFRESH,Medium,62.685313,Traffic decline is seasonal or the content was...
26840,content_7f116ae1f6f5,Refresh Content,STALE_REFRESH,Medium,62.651968,Traffic decline is seasonal or the content was...
1227,content_4f241bad48a3,Refresh Content,STALE_REFRESH,Medium,60.758150,Traffic decline is seasonal or the content was...
16475,content_f2b4acf220d9,Refresh Content,STALE_REFRESH,Medium,58.622681,Traffic decline is seasonal or the content was...
23506,content_57d8360dee59,Refresh Content,STALE_REFRESH,Medium,58.622681,Traffic decline is seasonal or the content was...
1147,content_ab27c30d81f4,Refresh Content,STALE_REFRESH,Medium,58.525469,Traffic decline is seasonal or the content was...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:
print("Weak picks identified:")

weak = df[df["score"] < 40][
    [
        "content_id",
        "score",
        "reason_code",
        "action_label"
    ]
].head(10)

display(weak)

Weak picks identified:


,content_id,score,reason_code,action_label
0,content_304f48230142,35.944054,CTR_FIX,Improve Title & Meta
1,content_a1fb4e703a9e,18.141250,LOW_PRIORITY,Monitor
2,content_9aa793d4d895,17.957676,LOW_PRIORITY,Monitor
3,content_331d6c4de07b,13.398722,LOW_PRIORITY,Monitor
4,content_d99b7a2d90ca,14.770673,LOW_PRIORITY,Monitor
5,content_d4084a4bc775,36.006919,QUICK_WIN,Optimise Existing Content
6,content_9a34b442b552,21.106676,LOW_PRIORITY,Monitor
7,content_a63219c6e95a,12.257667,CTR_FIX,Improve Title & Meta
8,content_5e6c160719bc,17.747676,LOW_PRIORITY,Monitor
9,content_c27558df2b0c,22.662713,LOW_PRIORITY,Monitor


In [20]:
print("Leakage Check")

future_columns = [
    c for c in df.columns
    if "label" in c.lower()
    or "future" in c.lower()
    or "target" in c.lower()
]

print("Potential future/label columns:", future_columns)

if len(future_columns) == 0:
    print("✅ No future-window or label-derived inputs detected.")
else:
    print("⚠ Review these columns before using them.")

Leakage Check
Potential future/label columns: ['action_label']
⚠ Review these columns before using them.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.